In [53]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
import functools
from concurrent.futures import ThreadPoolExecutor
import time
import pyodbc
import pandas as pd
import os, json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import requests
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from tqdm import tqdm


In [54]:
import pyodbc

SERVER = "tcp:swp391-sp26-server.database.windows.net"
DATABASE = "SWP391_SP26_Group1"
USERNAME = "SWP391_SP26_Group1"  # đổi thành user admin Azure SQL của bạn (vd: TeamAdmin hoặc adminuser)
PASSWORD = "Team11111111"

def get_conn():
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={SERVER},1433;"
        f"DATABASE={DATABASE};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
    )
    return pyodbc.connect(conn_str)

conn = get_conn()
cursor = conn.cursor()
cursor.execute("SELECT TOP 5 * FROM dbo.SystemErrorLog;")
for row in cursor.fetchall():
    print(row)


('0e0c3957-ee9e-438d-bd3a-904b50076d5c', None, 'Exception', 'Test hê thống Log lỗi AEMS', '   at AEMS_Solution.Controllers.HomeController.Index() in D:\\FPT\\2026 SPRING\\SWP391\\SWP391_SP26\\AEMS_Solution\\AEMS_Solution\\Controllers\\HomeController.cs:line 20\r\n   at lambda_method26(Closure, Object, Object[])\r\n   at Microsoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionContext actionContext, IActionResultTypeMapper mapper, ObjectMethodExecutor executor, Object controller, Object[] arguments)\r\n   at Microsoft.AspNetCore.Mvc.Infrastructure.ControllerActionInvoker.InvokeActionMethodAsync()\r\n   at Microsoft.AspNetCore.Mvc.Infrastructure.ControllerActionInvoker.Next(State& next, Scope& scope, Object& state, Boolean& isCompleted)\r\n   at Microsoft.AspNetCore.Mvc.Infrastructure.ControllerActionInvoker.InvokeNextActionFilterAsync()\r\n--- End of stack trace from previous location ---\r\n   at Microsoft.AspNetCore.Mvc.Infrastructure.Control

In [55]:
from concurrent.futures import ThreadPoolExecutor

In [56]:
#Knowledge Base

In [57]:

def load_events_bundle():
    

    # 1) Event
    events = pd.read_sql("""
        SELECT
            e.Id AS EventId,
            e.Title,
            e.Description,
            e.Location,
            e.StartTime,
            e.EndTime,
            e.Status,
            e.DepartmentId,
            e.SemesterId,
            e.UpdatedAt
        FROM Event e
        WHERE (e.DeletedAt IS NULL OR e.DeletedAt = '') 
          AND e.Status IN ('Published')  
    """, get_conn())

    # 2) EventAgenda
    agenda = pd.read_sql("""
        SELECT
            a.Id AS AgendaId,
            a.EventId,
            a.SessionName,
            a.Description AS SessionDescription,
            a.SpeakerName,
            a.StartTime AS SessionStart,
            a.EndTime AS SessionEnd
        FROM EventAgenda a
        WHERE (a.DeletedAt IS NULL OR a.DeletedAt = '')
    """, get_conn())

    # 3) EventDocuments
    docs = pd.read_sql("""
        SELECT
            d.Id AS DocId,
            d.EventId,
            d.Name AS DocTitle,
            d.Url
        FROM EventDocument d
        WHERE (d.DeletedAt IS NULL OR d.DeletedAt = '')
    """, get_conn())
    logs = pd.read_sql("""
        SELECT
            sel.Id AS LogId,
            sel.StatusCode,
            sel.ExceptionType,
            sel.ExceptionMessage,
            sel.StackTrace,
            sel.Source,
            sel.UserId,
            sel.CreatedAt
        FROM SystemErrorLog sel
        WHERE sel.CreatedAt IS NOT NULL
    """, get_conn())

    get_conn().close()
    return events, agenda, docs,logs


In [58]:
import sys
print(sys.executable)


d:\PROJECT_SWP_SP26\Python\.venv-rag\Scripts\python.exe


In [59]:
#Build event context
def build_event_text(event_row, agenda_rows, doc_rows):
    
    lines = []
    lines.append(f"EVENT OVERVIEW")
    lines.append(f"Title: {event_row['Title']}")
    lines.append(f"Description: {event_row.get('Description','')}")
    lines.append(f"Location: {event_row.get('Location','')}")
    lines.append(f"Time: {event_row.get('StartTime','')} - {event_row.get('EndTime','')}")
    lines.append(f"Status: {event_row.get('Status','')}")
    lines.append("")

    if len(agenda_rows) > 0:
        lines.append("AGENDA")
        for _, a in agenda_rows.iterrows():
            lines.append(f"- Session: {a.get('SessionName','')}")
            lines.append(f"  Speaker: {a.get('SpeakerName','')}")
            lines.append(f"  Time: {a.get('SessionStart','')} - {a.get('SessionEnd','')}")
            lines.append(f"  Detail: {a.get('SessionDescription','')}")
        lines.append("")

    if len(doc_rows) > 0:
        lines.append("DOCUMENTS")
        for _, d in doc_rows.iterrows():
            lines.append(f"- {d.get('DocTitle','')}: {d.get('DocDescription','')}")
            if d.get("Url"):
                lines.append(f"  URL: {d.get('Url')}")
        lines.append("")

    return "\n".join(lines)

In [60]:
#Build KB logs
def build_systemlog_text(r):
    return "\n".join([
        "SYSTEM ERROR LOG",
        f"StatusCode: {r.get('StatusCode','')}",
        f"ExceptionType: {r.get('ExceptionType','')}",
        f"ExceptionMessage: {r.get('ExceptionMessage','')}",
        f"Source: {r.get('Source','')}",
        f"UserId: {r.get('UserId','')}",
        f"CreatedAt: {r.get('CreatedAt','')}",
        f"StackTrace: {r.get('StackTrace','')}",
    ])

In [61]:
#chunking context
def chunk_text(text: str, max_chars=3000, overlap=250):
    # đơn giản theo ký tự; bạn có thể thay bằng tokenizer theo token
    chunks = []
    i = 0
    while i < len(text):
        j = min(len(text), i + max_chars)
        chunks.append(text[i:j])
        i = j - overlap
        if i < 0:
            i = 0
        if j == len(text):
            break
    return chunks

In [62]:
#call LLMs
model = SentenceTransformer("intfloat/multilingual-e5-base", trust_remote_code=False)  # multilingual tốt cho VN+EN
index = faiss.read_index("kb/faiss.index")

d:\PROJECT_SWP_SP26\Python\.venv-rag\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [63]:
def main():
    events, agenda, docs,logs = load_events_bundle()


    all_chunks = []
    all_meta = []
    #for event
    for _, e in events.iterrows():
        eid = e["EventId"]
        a_rows = agenda[agenda["EventId"] == eid]
        d_rows = docs[docs["EventId"] == eid]
        

        doc_text = build_event_text(e, a_rows, d_rows)
        chunks = chunk_text(doc_text)

        for idx, ch in enumerate(chunks):
            meta = {
                "type" : "event",
                "access": "user", #admin và user đều xem được
                "event_id": str(eid),
                "chunk_id": idx,
                "department_id": str(e.get("DepartmentId","")),
                "semester_id": str(e.get("SemesterId","")),
                "status": str(e.get("Status","")),
                "updated_at": str(e.get("UpdatedAt","")),
                "source": "event_bundle"
            }
            all_chunks.append(ch)
            all_meta.append(meta)
    #for admin
    for _, l in logs.iterrows():
        log_text = build_systemlog_text(l)
        chunks = chunk_text(log_text, max_chars=500, overlap=100)

        for idx, ch in enumerate(chunks):
            all_chunks.append(ch)
            all_meta.append({
                "type": "system_log",
                "access": "admin",          # CHỐT: chỉ admin
                "log_id": str(l.get("LogId","")),
                "status_code": str(l.get("StatusCode","")),
                "exception_type": str(l.get("ExceptionType","")),
                "source": str(l.get("Source","")),
                "user_id": str(l.get("UserId","")),
                "created_at": str(l.get("CreatedAt","")),
        })

    # Embedding
    # Với e5: nên prefix "query:" / "passage:" cho đúng convention
    passages = [f"passage: {t}" for t in all_chunks]
    embs = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
    embs = np.array(embs, dtype="float32")

    # FAISS index
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine similarity nếu normalize
    index.add(embs)

    os.makedirs("kb", exist_ok=True)
    faiss.write_index(index, "kb/faiss.index")

    # lưu chunks + metadata để lookup
    with open("kb/chunks.jsonl", "w", encoding="utf-8") as f:
        for t, m in zip(all_chunks, all_meta):
            f.write(json.dumps({"text": t, "meta": m}, ensure_ascii=False) + "\n")

    print(f"KB built: {len(all_chunks)} chunks")
    return all_chunks, all_meta

all_chunks, all_meta = main()


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27968\2488093878.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27968\2488093878.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  agenda = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27968\2488093878.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27968\2488093878.py:46: UserWarning: pandas only supports SQLAlchemy co

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

KB built: 29 chunks


In [64]:
#Log
for i, m in enumerate(all_meta):
    if m["type"] == "system_log":
        print("META:", m)
        print("TEXT:")
        print(all_chunks[i][:500])
        break


META: {'type': 'system_log', 'access': 'admin', 'log_id': '0e0c3957-ee9e-438d-bd3a-904b50076d5c', 'status_code': 'None', 'exception_type': 'Exception', 'source': '/', 'user_id': 'None', 'created_at': '2026-01-22 18:14:26.190273'}
TEXT:
SYSTEM ERROR LOG
StatusCode: None
ExceptionType: Exception
ExceptionMessage: Test hê thống Log lỗi AEMS
Source: /
UserId: None
CreatedAt: 2026-01-22 18:14:26.190273
StackTrace:    at AEMS_Solution.Controllers.HomeController.Index() in D:\FPT\2026 SPRING\SWP391\SWP391_SP26\AEMS_Solution\AEMS_Solution\Controllers\HomeController.cs:line 20
   at lambda_method26(Closure, Object, Object[])
   at Microsoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionCont


In [65]:
print("Total chunks:", len(all_chunks))
print("Sample meta:", all_meta[0])
print("Sample chunk:\n", all_chunks[0][:1200])


Total chunks: 29
Sample meta: {'type': 'system_log', 'access': 'admin', 'log_id': '0e0c3957-ee9e-438d-bd3a-904b50076d5c', 'status_code': 'None', 'exception_type': 'Exception', 'source': '/', 'user_id': 'None', 'created_at': '2026-01-22 18:14:26.190273'}
Sample chunk:
 SYSTEM ERROR LOG
StatusCode: None
ExceptionType: Exception
ExceptionMessage: Test hê thống Log lỗi AEMS
Source: /
UserId: None
CreatedAt: 2026-01-22 18:14:26.190273
StackTrace:    at AEMS_Solution.Controllers.HomeController.Index() in D:\FPT\2026 SPRING\SWP391\SWP391_SP26\AEMS_Solution\AEMS_Solution\Controllers\HomeController.cs:line 20
   at lambda_method26(Closure, Object, Object[])
   at Microsoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionCont


In [66]:
def can_access(meta:dict, role: str):
    role = (role or "user").lower()
    access = (meta.get("access") or "user").lower()

    if role == "admin":
        return True

    # user chỉ xem access=user
    return access == "user"


In [67]:
import json

chunks = []
with open("kb/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print("len(chunks) =", len(chunks))
print("sample:", chunks[0].keys())  # phải có "text" và "meta"


len(chunks) = 29
sample: dict_keys(['text', 'meta'])


In [68]:
def retrieve(question: str, top_k: int = 5, role: str = "user"):
    q = model.encode([f"query: {question}"], normalize_embeddings=True)
    q = np.array(q, dtype="float32")

    # lấy dư để còn lọc quyền
    k = min(max(top_k * 5, top_k), index.ntotal)
    scores, ids = index.search(q, k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0 or idx >= len(chunks):
            continue

        item = chunks[idx]          # ✅ lấy item trước
        meta = item["meta"]         # ✅ lấy meta

        if not can_access(meta, role):
            continue

        results.append({
            "score": float(score),
            "text": item["text"],
            "meta": meta
        })

        if len(results) >= top_k:   # ✅ đủ top_k thì dừng
            break

    return results


In [69]:
class AskReq(BaseModel):
    question: str
    top_k: int = 5
    role: str = "user"


In [70]:
from fastapi.responses import StreamingResponse
app=FastAPI()
@app.post("/ask_stream")
def ask_stream(req: AskReq):
    hits = retrieve(req.question, req.top_k)

    context_blocks = []
    for h in hits:
        m = h["meta"]
        context_blocks.append(
           f"[chunk={m.get('chunk_id')}]\n{h['text']}"
                               )
    context = "\n\n---\n\n".join(context_blocks)

    prompt = f"""Bạn là trợ lý của hệ thống AEMS.
CHỈ được dùng thông tin trong CONTEXT. Nếu thiếu thì nói: "Chưa có dữ liệu trong hệ thống để trả lời câu này."

CONTEXT:
{context}

QUESTION:
{req.question}

Trả lời theo cách của bạn và không được bịa thông tin.
"""


In [71]:
#demo

In [72]:
for i, (t, m) in enumerate(zip(all_chunks, all_meta)):
    print("="*80)
    print("CHUNK", i)
    print("TYPE:", m["type"])
    print("ACCESS:", m["access"])
    print("ID:", m.get("event_id") or m.get("log_id"))
    print("LEN:", len(t))
    print(t[:500])  # in 500 ký tự cho đỡ dài

    if i >= 3:
        break


CHUNK 0
TYPE: system_log
ACCESS: admin
ID: 0e0c3957-ee9e-438d-bd3a-904b50076d5c
LEN: 500
SYSTEM ERROR LOG
StatusCode: None
ExceptionType: Exception
ExceptionMessage: Test hê thống Log lỗi AEMS
Source: /
UserId: None
CreatedAt: 2026-01-22 18:14:26.190273
StackTrace:    at AEMS_Solution.Controllers.HomeController.Index() in D:\FPT\2026 SPRING\SWP391\SWP391_SP26\AEMS_Solution\AEMS_Solution\Controllers\HomeController.cs:line 20
   at lambda_method26(Closure, Object, Object[])
   at Microsoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionCont
CHUNK 1
TYPE: system_log
ACCESS: admin
ID: 0e0c3957-ee9e-438d-bd3a-904b50076d5c
LEN: 500
osoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionContext actionContext, IActionResultTypeMapper mapper, ObjectMethodExecutor executor, Object controller, Object[] arguments)
   at Microsoft.AspNetCore.Mvc.Infrastructure.ControllerActionInvoker.InvokeActionMethodAsync()
   at 

In [73]:
query = "mô tả chi tiết E001                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     "
q_emb = model.encode([f"query: {query}"], normalize_embeddings=True)

D, I = index.search(np.array(q_emb, dtype="float32"), k=3)

for score, idx in zip(D[0], I[0]):
    print("SCORE:", score)
    print("META:", all_meta[idx])
    print(all_chunks[idx][:600])
    print("="*60)


SCORE: 0.8039056
META: {'type': 'system_log', 'access': 'admin', 'log_id': '0e0c3957-ee9e-438d-bd3a-904b50076d5c', 'status_code': 'None', 'exception_type': 'Exception', 'source': '/', 'user_id': 'None', 'created_at': '2026-01-22 18:14:26.190273'}
SYSTEM ERROR LOG
StatusCode: None
ExceptionType: Exception
ExceptionMessage: Test hê thống Log lỗi AEMS
Source: /
UserId: None
CreatedAt: 2026-01-22 18:14:26.190273
StackTrace:    at AEMS_Solution.Controllers.HomeController.Index() in D:\FPT\2026 SPRING\SWP391\SWP391_SP26\AEMS_Solution\AEMS_Solution\Controllers\HomeController.cs:line 20
   at lambda_method26(Closure, Object, Object[])
   at Microsoft.AspNetCore.Mvc.Infrastructure.ActionMethodExecutor.SyncActionResultExecutor.Execute(ActionCont
SCORE: 0.80373704
META: {'type': 'system_log', 'access': 'admin', 'log_id': 'be6af5b3-7043-4791-92e8-704af5b13ae5', 'status_code': 'None', 'exception_type': 'Exception', 'source': '/', 'user_id': 'None', 'created_at': '2026-01-22 18:14:45.036596'}
SYSTEM

In [74]:
import os
os.environ["GROQ_API_KEY"] = "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"


In [75]:
import os, requests

def call_llm_groq(question: str, hits: list, model="llama-3.1-8b-instant", max_context_chars=6000):
    # gom context từ hits
    blocks = []
    total = 0
    for h in hits:
        block = f"[chunk={h['meta'].get('chunk_id')}]\n{h['text']}"
        if total + len(block) > max_context_chars:
            break
        blocks.append(block)
        total += len(block)

    context = "\n\n---\n\n".join(blocks)

    prompt = f"""Bạn là trợ lý của hệ thống AEMS.
CHỈ được dùng thông tin trong CONTEXT. Nếu thiếu thì nói: "Chưa có dữ liệu trong hệ thống để trả lời câu này.
Tuyệt đối không đưa đưa bất kì thông tin giả nào mà chưa có cơ sở"

CONTEXT:
{context}

QUESTION:
{question}


"""

    api_key = "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" 
    if not api_key:
        raise ValueError("Thiếu biến môi trường GROQ_API_KEY")

    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}],
        "temperature": 0.2,
        "stream":True,
    }

    with requests.post(url, headers=headers, json=payload, stream=True, timeout=60) as r:
        r.encoding = 'utf-8'  
        r.raise_for_status()

        # Groq trả về dạng SSE: mỗi dòng bắt đầu bằng "data: ..."
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            if line.startswith("data: "):
                data = line[len("data: "):].strip()
            else:
                continue

            if data == "[DONE]":
                break

            try:
                obj = json.loads(data)
                delta = obj["choices"][0]["delta"]
                token = delta.get("content")
                if token:
                    yield token
            except Exception:
                # bỏ qua line lỗi parse
                continue


In [76]:
print("index.ntotal =", index.ntotal)
print("len(chunks)  =", len(chunks))

# sanity: in thử 2 dòng đầu file
import itertools, json
with open("kb/chunks.jsonl", "r", encoding="utf-8") as f:
    first2 = list(itertools.islice(f, 2))
print("first line sample:", first2[0][:200])


index.ntotal = 29
len(chunks)  = 29
first line sample: {"text": "SYSTEM ERROR LOG\nStatusCode: None\nExceptionType: Exception\nExceptionMessage: Test hê thống Log lỗi AEMS\nSource: /\nUserId: None\nCreatedAt: 2026-01-22 18:14:26.190273\nStackTrace:    at 


In [79]:
q = "Hệ thống hay lỗi gì?"
hits_user  = retrieve(q, top_k=5, role="user")
hits_admin = retrieve(q, top_k=5, role="admin")

print("USER ANSWER:")
for tok in call_llm_groq(q, hits_user):   # call_llm_groq đang yield token
    print(tok, end="", flush=True)

print("\n\nADMIN ANSWER:")
for tok in call_llm_groq(q, hits_admin):
    print(tok, end="", flush=True)


USER ANSWER:
Chưa có dữ liệu trong hệ thống để trả lời câu này.

ADMIN ANSWER:
Hệ thống AEMS có lỗi sau:

1. Lỗi trong Home Controller: Có 2 lỗi xảy ra tại Home Controller vào ngày 22/01/2026, lần lượt là vào lúc 18:14:45 và 18:14:26. Lỗi này được gây ra bởi việc gọi đến method Index() tại HomeController.cs, nhưng không có thông tin chi tiết về lỗi này.

2. Lỗi đăng nhập: Có 1 lỗi xảy ra tại AuthController vào ngày 22/01/2026, lúc 18:54:20. Lỗi này được gây ra bởi việc đăng nhập với email hoặc mật khẩu không chính xác hoặc tài khoản bị khóa.

3. Lỗi xác thực Google: Có 2 lỗi xảy ra tại /signin-google vào ngày 23/01/2026, lần lượt là vào lúc 10:41:45 và 10:38:16. Lỗi này được gây ra bởi việc truy cập vào tài khoản Google bị từ chối.

In [78]:
import requests

r = requests.post(
    "http://127.0.0.1:8000/ask_stream",
    json={"question": "Sự kiện E001 nói về gì?", "top_k": 5},
    stream=True
)

for chunk in r.iter_content(chunk_size=None, decode_unicode=True):
    if chunk:
        print(chunk, end="", flush=True)


ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /ask_stream (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))